# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, where (in z, µm) real tissue signal actually starts and ends, using one already-finished round (default: `cells`).

Follows the architecture rules in [`NOTEBOOK_GUIDELINES.md`](../../NOTEBOOK_GUIDELINES.md) (repo root) throughout -- every nontrivial step below is a **calculation cell** (cached under `analysis/cache/measure_tissue_thickness/`, with `ProgressReporter` progress, skipping recomputation when its cache still matches the current inputs) followed by a separate **display cell** (plot or printed summary).

Procedure:
1. Resolve the target round's frame table and `CHANNEL_NM`'s z-steps.
2. For every FOV, read every z-plane of `CHANNEL_NM` (still far fewer than the round's full multi-color frame count) and build an EXACT, bin-width-1 histogram of each frame -- a true Counter over observed pixel intensities (`analysis.fov.compute_channel_counters`, stored sparsely via `numpy.unique`, cached per FOV). Exact per-intensity counts mean every later step (reference-frame selection, threshold estimation, the per-z true-pixel-count profile) is derived from this one cached read, without recomputing or re-reading pixels. This is the reference cell for the caching + progress-reporting pattern used everywhere else in the notebook.
3. Across every FOV and z, find the frame with the **highest mean intensity** (a visual "what does real tissue look like" reference) and the `N_BACKGROUND_FRAMES` frames with the **lowest mean intensity** (the best available proxy for pure background/no-tissue signal). Display both, overlay all the background frames' histograms plus the tissue frame's histogram, and derive `THRESHOLD` as the highest pixel value observed among those background frames -- i.e. the highest pixel value that can plausibly occur as noise, bounded only by frames confidently known to be background. (An earlier version instead picked the value that best *separated* one background frame from one tissue frame; that was rejected because it can still misclassify a genuinely empty frame as tissue whenever its own noise tail crosses the separating value -- see section 5.) Review the plot and override `THRESHOLD` manually if it looks wrong.
4. For every FOV, derive its true-pixel-count (NTP) profile directly from its cached Counter (no further disk read). Report the shallowest (`z_first_um`) and deepest (`z_last_um`) z with signal (NTP > `NTP_THRESHOLD`), plus `is_contiguous` (whether signal held continuously in between). Both boundaries matter: some FOVs are blank at the top of the imaged range and only pick up signal partway down, not just "signal that eventually stops".
5. Lay every FOV's `z_first_um`/`z_last_um` out on its stage-position grid and plot as heatmaps.
6. Verify `z_last_um` visually: render the actual frame at each FOV's own last-passing z and tile them into one mosaic (section 10) -- every tile should look like a real tissue edge, not blank/noise or clearly mid-tissue.

Figures and results are saved under `SAMPLE_DIR/analysis/figures/` and `SAMPLE_DIR/analysis/` respectively, in addition to being shown inline.

Runs anywhere the standard `SAMPLE_DIR/{data,metadata,positions,analysis}` layout is reachable, including a cluster node (same convention as `05_batch_sample_review.ipynb`/`07_cluster_submit_analysis.ipynb`). Step 2's backfill loop is intentionally sequential, not process-pool-parallelized: on a shared SLURM node, `os.cpu_count()` reports the node's total core count, not this job's actual memory allocation, so sizing a worker pool off it (`config.resolved_n_workers`) can spawn far more workers than the job's real memory allows -- each holding a stack in memory at once -- and get OOM-killed (`BrokenProcessPool`). Reading only `CHANNEL_NM`'s frames (not the whole multi-color stack) keeps the sequential version fast without a pool.

## 1 — Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.transform import resize as sk_resize

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import iter_image_frames, path_mtime
from MERci.analysis.fov       import (
    compute_channel_counters, save_channel_counters, load_channel_counters,
    counter_mean, counter_percentile, rebin_counter, ntp_profile_from_counters,
)
from MERci.analysis.round        import create_mosaic
from MERci.visualization         import display_mosaic
from MERci.scheduler             import resolve_round_flip_y
from MERci.acquisition.configs   import find_frame_table_for_hal_config, read_hal_exposure_time
from MERci.acquisition.merlin_config import load_microscope_orientation, apply_microscope_orientation

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which microscope this experiment was acquired on -- must match a filename
# in MERci.acquisition.merlin_config.resolve_microscope_parameters_filename
# (e.g. "ST2" -> data/configs/merlin/microscope/STORM2FUSION_2304_60xSil.json).
# Used for the last-passing-z verification mosaic (section 10): different
# microscopes need different flip_horizontal/flip_vertical/transpose camera
# corrections to display FOVs in the correct real-world orientation.
MICROSCOPE = "ST2"

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# A z-plane still counts as "has tissue" if its true-pixel count (NTP) exceeds this.
NTP_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Display-histogram resolution (section 5) -- these are re-binned on demand from the
# exact per-intensity Counter, so changing this never requires recomputing anything.
DISPLAY_HIST_BINS      = 200
LINEAR_HIST_PERCENTILE = 99.0   # upper bound of the linear-scale display histogram

# How many of the lowest-mean-intensity frames (across every FOV/z) to treat as
# "confidently background" for threshold estimation (section 5), and what
# percentile of each background frame's own pixel distribution to take as its
# noise ceiling (100 = literal max pixel value observed in that frame). A single
# lowest-mean frame can have an atypical noise ceiling (one hot pixel, one dead
# pixel); pooling N=10 frames and overlaying their histograms lets THRESHOLD be
# picked visually, above the point where none of them still have real mass --
# see section 5. Two-class separation between one "background" and one "tissue"
# frame (an earlier version of this notebook) was tried and rejected: it can
# still call a genuinely empty frame "tissue" whenever that frame's own noise
# tail crosses the separating value, since that criterion balances false
# positives against false negatives on BOTH classes instead of bounding the
# background side alone.
N_BACKGROUND_FRAMES  = 10
BACKGROUND_PERCENTILE = 100.0

# Binarization intensity threshold; None = auto-estimate as the highest pixel
# value observed across the N_BACKGROUND_FRAMES lowest-mean frames (section 5)
# -- review that plot before trusting the estimate on a new experiment.
THRESHOLD = None

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- matplotlib's default
# sizes shrink relative to figsize, so a wide/short figure (section 7's FOV-grid
# heatmap) reads far smaller than a square one (section 5's reference-frame
# figure) even at the same nominal font size. Every plotting cell in this
# notebook sets these explicitly instead of relying on the default.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Microscope         : {MICROSCOPE}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# NOTEBOOK_GUIDELINES.md #2/#3: every calculation cell below caches its result
# under analysis/cache/<notebook_name>/ and skips recomputation when a valid
# cache is already there -- deliberately NOT tracker.histogram_path()'s
# canonical location, since FOVScheduler's own per-frame histograms are
# fixed-width (lossy) 512-bin ones that can't be turned back into an EXACT
# per-intensity Counter.
NOTEBOOK_NAME = "measure_tissue_thickness"
cache_dir     = config.analysis_dir / "cache" / NOTEBOOK_NAME
channel_counters_dir = cache_dir / "channel_counters"
channel_counters_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Compute (or load cached) exact per-z Counter histograms for `CHANNEL_NM`

For every FOV, reads every z-plane of `CHANNEL_NM` (still far fewer than the round's
full multi-color frame count) and builds an EXACT, bin-width-1 histogram of each
frame -- a true Counter over observed pixel intensities (`analysis.fov.
compute_channel_counters`, stored sparsely: only intensity values that actually
occur, via `numpy.unique`), not a fixed-bin-count histogram. Having every
intensity's exact count means the reference-frame selection (section 5), the
threshold-estimation display histograms (section 5), and the per-z true-pixel-count
profile (section 6) can ALL be derived from this ONE cached read, without
re-reading pixels or recomputing anything.

No fallback to an existing full per-frame histogram here (unlike earlier versions
of this notebook): `FOVScheduler`'s own histograms are fixed-width (lossy) 512-bin
ones, which can't be turned back into an exact per-intensity Counter -- every FOV's
Counters are computed by, and cached under, this notebook alone
(`analysis/cache/measure_tissue_thickness/channel_counters/`).

This is the reference cell for `NOTEBOOK_GUIDELINES.md` #2-4: per-FOV caching
(only what's actually missing gets computed), and `ProgressReporter`-driven
progress on the remaining work.

In [ ]:
def channel_counters_path(fpath):
    return channel_counters_dir / f"{Path(fpath).stem}_counters.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

channel_counters = {}   # fov_id -> compute_channel_counters()-shaped dict
from_own_cache, to_compute = [], []
n_missing_on_disk = 0

for fpath in files:
    if channel_counters_path(fpath).exists():
        from_own_cache.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    channel_counters[meta.fov_id_of_file(fpath)] = load_channel_counters(channel_counters_path(fpath))
print(f"{len(from_own_cache)} channel Counter(s) already cached -- loaded directly.")

n_computed = 0
if to_compute:
    print(f"Computing {len(to_compute)} missing channel Counter(s) sequentially "
          f"(all {len(z_frame_indices)} z-step(s) of channel {CHANNEL_NM:.0f} nm per FOV).")
    reporter = ProgressReporter(total=len(to_compute), label="Computing channel Counters")
    for fpath in reporter.wrap(to_compute):
        counters = compute_channel_counters(
            fpath, z_frame_indices,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        save_channel_counters(channel_counters_path(fpath), counters)
        channel_counters[meta.fov_id_of_file(fpath)] = counters
        n_computed += 1

print(f"Channel Counters ready for {len(channel_counters)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Reference frames: highest-mean (tissue) + lowest-mean (background) frames

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Across every FOV and z, finds the frame with the highest mean intensity (a
visual "what does real tissue look like" reference) and the
`N_BACKGROUND_FRAMES` frames with the lowest mean intensity (the best
available proxy for pure background/no-tissue signal) -- not a fixed z
pooled across FOVs (an earlier version's approach, which produced a poor,
non-bimodal pooled histogram since that fixed z is blank for some FOVs --
see section 6's `z_first_um`).

`THRESHOLD` is auto-estimated as the highest pixel value observed among
those `N_BACKGROUND_FRAMES` background frames -- the highest pixel value
that can plausibly occur as background noise, derived only from frames
confidently known to be background. An earlier version of this notebook
instead picked the value that best *separated* one background frame from
one tissue frame (`analysis.fov.two_class_separating_threshold`, since
removed): that criterion balances false positives against false negatives
across *both* classes, so a genuinely empty frame could still have its own
noise tail cross the separating value and get counted as tissue. Bounding
only the background side avoids that failure mode. Review the overlaid
histogram plot before trusting the estimate -- if it looks wrong, set
`THRESHOLD` by hand in section 2 and re-run from here.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached (NOTEBOOK_GUIDELINES.md #2/#3): rerunning this cell to tweak the
# plot in the next cell never rescans every FOV x z combination again, as
# long as the same set of Counters (n_frames_considered) is still current.
reference_frames_cache = cache_dir / f"reference_frames_round{target_round_id}.npz"
n_frames_now = sum(len(c["values_per_z"]) for c in channel_counters.values())

cached = None
if reference_frames_cache.exists():
    cached = np.load(reference_frames_cache)
    if int(cached["n_frames_considered"]) != n_frames_now:
        print(f"Cached reference-frame selection is stale "
              f"({int(cached['n_frames_considered'])} vs. {n_frames_now} frame(s) now available) -- recomputing.")
        cached = None

if cached is not None:
    n_frames_considered = int(cached["n_frames_considered"])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = (
        float(cached["best_mean"]), int(cached["best_fov_id"]), int(cached["best_pos"]),
        int(cached["best_frame_idx"]), float(cached["best_z_um"]),
    )
    worst_fov_ids, worst_pos, worst_frame_idx, worst_z_um = (
        cached["worst_fov_ids"], cached["worst_pos"], cached["worst_frame_idx"], cached["worst_z_um"],
    )
    print(f"Loaded cached reference-frame selection ({reference_frames_cache.name}) -- "
          f"{n_frames_considered} FOV x z combination(s), matches current data.")
else:
    frame_records = []   # (mean, fov_id, pos_in_z, frame_idx, z_um)
    reporter = ProgressReporter(total=len(channel_counters), label="Scanning frame means")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        for pos, (values, counts) in enumerate(zip(counters["values_per_z"], counters["counts_per_z"])):
            mean = counter_mean(values, counts)
            frame_records.append((mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos])))

    n_frames_considered = len(frame_records)
    frame_records.sort(key=lambda r: r[0])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = frame_records[-1]
    worst_records = frame_records[:N_BACKGROUND_FRAMES]

    worst_fov_ids   = np.array([r[1] for r in worst_records], dtype=np.int64)
    worst_pos       = np.array([r[2] for r in worst_records], dtype=np.int64)
    worst_frame_idx = np.array([r[3] for r in worst_records], dtype=np.int64)
    worst_z_um      = np.array([r[4] for r in worst_records], dtype=np.float64)

    np.savez_compressed(
        reference_frames_cache,
        n_frames_considered=n_frames_considered,
        best_mean=best_mean, best_fov_id=best_fov_id, best_pos=best_pos,
        best_frame_idx=best_frame_idx, best_z_um=best_z_um,
        worst_fov_ids=worst_fov_ids, worst_pos=worst_pos,
        worst_frame_idx=worst_frame_idx, worst_z_um=worst_z_um,
    )
    print(f"Scanned {n_frames_considered} FOV x z combination(s); cached selection to {reference_frames_cache.name}.")

best_values, best_counts = (channel_counters[best_fov_id]["values_per_z"][best_pos],
                            channel_counters[best_fov_id]["counts_per_z"][best_pos])
worst_value_counts = [
    (channel_counters[int(fov_id)]["values_per_z"][int(pos)], channel_counters[int(fov_id)]["counts_per_z"][int(pos)])
    for fov_id, pos in zip(worst_fov_ids, worst_pos)
]

# Automatic THRESHOLD estimate: the highest pixel value observed among the
# N_BACKGROUND_FRAMES lowest-mean frames (BACKGROUND_PERCENTILE-th percentile
# of each, default 100 = literal max) -- "the highest pixel value that can
# occur in background noise", derived only from frames confidently known to
# be background (see the N_BACKGROUND_FRAMES/BACKGROUND_PERCENTILE comment in
# section 2 for why the earlier two-class separating threshold was rejected).
estimated_threshold = float(max(
    counter_percentile(values, counts, BACKGROUND_PERCENTILE) for values, counts in worst_value_counts
))

print(f"Highest-mean frame: FOV {best_fov_id}, frame_idx={best_frame_idx}, z={best_z_um:.2f} um (mean {best_mean:.0f})")
print(f"Lowest-mean {len(worst_fov_ids)} frame(s) (background reference):")
for fov_id, frame_idx, z_um in zip(worst_fov_ids, worst_frame_idx, worst_z_um):
    print(f"  FOV {fov_id}, frame_idx={frame_idx}, z={z_um:.2f} um")
print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of {len(worst_value_counts)} "
      f"background frame(s)): {estimated_threshold:.0f}")

In [ ]:
# ---- Display --------------------------------------------------------------
# Counters have no spatial information -- re-read the tissue reference frame
# and the single emptiest background frame to display them.
best_fpath  = next(f for f in files if meta.fov_id_of_file(f) == best_fov_id)
worst_fpath = next(f for f in files if meta.fov_id_of_file(f) == int(worst_fov_ids[0]))
best_frame  = next(frame for _, frame in iter_image_frames(
    best_fpath, [best_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))
worst_frame = next(frame for _, frame in iter_image_frames(
    worst_fpath, [int(worst_frame_idx[0])], frame_width=config.frame_width, frame_height=config.frame_height,
))

fig_img, axes_img = plt.subplots(1, 2, figsize=(10, 5))
for ax, frame, fov_id, z_um, title in zip(
    axes_img, (worst_frame, best_frame), (int(worst_fov_ids[0]), best_fov_id), (float(worst_z_um[0]), best_z_um),
    ("Lowest-mean frame (background)", "Highest-mean frame (tissue)"),
):
    im = ax.imshow(frame, cmap="gray")
    ax.set_title(f"{title}\nFOV {fov_id}, z={z_um:.1f} um", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig_img.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig_img.tight_layout()
fig_img.savefig(figures_dir / f"tissue_thickness_reference_frames_round{target_round_id}.png", dpi=150)
plt.show()

# Overlay every background frame's histogram (thin lines) + the tissue
# reference frame's histogram (bold) -- same idiom as
# acquisition.mosaic.plot_tile_intensity_histograms's "thin per-tile lines +
# one bold reference line". Log-scale (full range) and linear-scale (combined
# min -> the tissue frame's LINEAR_HIST_PERCENTILE), both re-binned on demand
# from the exact Counters -- no raw pixel re-read for either.
combined_min = float(min(min(v.min() for v, _ in worst_value_counts), best_values.min()))
combined_max = float(max(max(v.max() for v, _ in worst_value_counts), best_values.max()))
pct_value    = counter_percentile(best_values, best_counts, LINEAR_HIST_PERCENTILE)

log_edges       = np.logspace(np.log10(max(combined_min, 1)), np.log10(combined_max), DISPLAY_HIST_BINS + 1)
log_bin_centers = np.sqrt(log_edges[:-1] * log_edges[1:])   # geometric mean = correct center in log space

linear_edges       = np.linspace(combined_min, max(pct_value, combined_min + 1), DISPLAY_HIST_BINS + 1)
linear_bin_centers = 0.5 * (linear_edges[:-1] + linear_edges[1:])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for values, counts in worst_value_counts:
    axes[0].plot(log_bin_centers, rebin_counter(values, counts, log_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
    axes[1].plot(linear_bin_centers, rebin_counter(values, counts, linear_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].plot(log_bin_centers, rebin_counter(best_values, best_counts, log_edges), "-", color="darkorange", lw=1.8)
axes[1].plot(linear_bin_centers, rebin_counter(best_values, best_counts, linear_edges), "-", color="darkorange", lw=1.8)
# One labeled proxy line per style -- a legend entry per background line would repeat N_BACKGROUND_FRAMES times.
for ax in axes:
    ax.plot([], [], "-", color="steelblue", alpha=0.5, lw=1.0,
            label=f"{len(worst_value_counts)} lowest-mean (background) frames")
    ax.plot([], [], "-", color="darkorange", lw=1.8, label=f"highest-mean frame (FOV {best_fov_id})")

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Log-scale (full range)", fontsize=PLOT_TITLE_FONTSIZE)

axes[1].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title(f"Linear-scale (min={combined_min:.0f} -> p{LINEAR_HIST_PERCENTILE:.0f}={pct_value:.0f})",
                   fontsize=PLOT_TITLE_FONTSIZE)

for ax in axes:
    ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
               label=f"background noise ceiling = {estimated_threshold:.0f}")
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- {len(worst_value_counts)} lowest-mean (background) vs. "
             f"highest-mean (tissue) frame histograms", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()
fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: true-pixel-count (NTP) profile, derived from cached Counters

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Purely in-memory: every FOV's exact per-z Counter is already cached/loaded from
section 4, so deriving each z's true-pixel count against `THRESHOLD`
(`analysis.fov.ntp_profile_from_counters`) needs no further disk read at all.
The resulting per-FOV table is itself cached (`NOTEBOOK_GUIDELINES.md` #2/#3),
keyed on `THRESHOLD`/`NTP_THRESHOLD`/FOV count so changing either one in
section 2 or 5 correctly invalidates it. Reports both `z_first_um`/`z_last_um`
(shallowest/deepest z with signal) and `is_contiguous` (`False` if signal
turned off and back on somewhere in between -- debris, folded tissue, noise)
-- some FOVs are blank at the top of the imaged range and only pick up tissue
signal partway down, so both boundaries matter, not just "signal that
eventually stops".

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached (NOTEBOOK_GUIDELINES.md #2/#3), keyed on THRESHOLD/NTP_THRESHOLD/FOV
# count -- any of those changing invalidates the cache and triggers a recompute.
results_cache      = cache_dir / f"ntp_profile_round{target_round_id}.csv"
results_meta_cache = cache_dir / f"ntp_profile_round{target_round_id}.json"
cache_signature     = {"threshold": float(THRESHOLD), "ntp_threshold": float(NTP_THRESHOLD),
                        "n_fovs": len(channel_counters)}

cached_signature = json.loads(results_meta_cache.read_text()) if results_meta_cache.exists() else None

if cached_signature == cache_signature and results_cache.exists():
    results_df = pd.read_csv(results_cache)
    print(f"Loaded cached NTP profile ({results_cache.name}) -- "
          f"THRESHOLD/NTP_THRESHOLD/FOV count unchanged since it was written.")
else:
    results = []
    reporter = ProgressReporter(total=len(channel_counters), label="Deriving NTP profiles")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        profile = ntp_profile_from_counters(counters, THRESHOLD, NTP_THRESHOLD)
        results.append({
            "fov_id":        fov_id,
            "z_first_um":    profile["z_first_um"],
            "z_last_um":     profile["z_last_um"],
            "is_contiguous": profile["is_contiguous"],
            "x_um":          meta.fovs[fov_id].position[0],
            "y_um":          meta.fovs[fov_id].position[1],
        })
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_cache, index=False)
    results_meta_cache.write_text(json.dumps(cache_signature))
    print(f"Derived NTP profile for {len(results_df)} FOV(s); cached to {results_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
n_no_signal      = results_df["z_last_um"].isna().sum()
n_not_contiguous = (~results_df["is_contiguous"]).sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above NTP_THRESHOLD at all; "
      f"{n_not_contiguous} had signal turn off and back on somewhere in between (is_contiguous=False).")
print("z_first_um:")
print(results_df["z_first_um"].describe())
print("z_last_um:")
print(results_df["z_last_um"].describe())

## 7 — Tissue-extent heatmaps across the FOV grid

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1),
with explicit plot font sizes (`NOTEBOOK_GUIDELINES.md` #5 -- this figure's
wide/short aspect ratio made matplotlib's default fonts read noticeably
smaller than section 5's square reference-frame figure, at the same nominal
size; `PLOT_TITLE_FONTSIZE`/`PLOT_LABEL_FONTSIZE`/`PLOT_TICK_FONTSIZE` fix
that).

Two panels sharing one color scale: where tissue signal **starts**
(`z_first_um` -- a shallow-imaged range wasted before signal appears would
show up here as bright patches) and where it **ends** (`z_last_um`, the
original "how deep does tissue go" question).

In [ ]:
# ---- Calculation --------------------------------------------------------
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    04_view_intensity_stats.ipynb's heatmap: round to the nearest integer micron, then
    rank each axis's unique values -- robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


def build_matrix(column, results_df, grid, n_x, n_y):
    matrix = np.full((n_y, n_x), np.nan)
    for _, row in results_df.iterrows():
        xi, yi = grid[row["fov_id"]]
        if pd.notna(row[column]):
            matrix[yi, xi] = row[column]
    return matrix


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1

z_first_matrix = build_matrix("z_first_um", results_df, grid, n_x, n_y)
z_last_matrix  = build_matrix("z_last_um",  results_df, grid, n_x, n_y)

print(f"FOV grid: {n_x} x {n_y} (columns x rows), {len(fov_ids)} FOV(s) placed.")

In [ ]:
# ---- Display --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(max(9, n_x * 0.7 + 3), max(4, n_y * 0.4 + 1.5)))
for ax, matrix, title in zip(
    axes,
    (z_first_matrix, z_last_matrix),
    ("z_first -- signal starts", "z_last -- signal ends"),
):
    im = ax.imshow(matrix, cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xlabel("X grid index  (increasing stage X →)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.04)
cbar.set_label("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle(f"Round {target_round_id} -- tissue extent map ({len(fov_ids)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

## 8 — Experimental vs. theoretical acquisition time per frame

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1),
with `ProgressReporter` progress over the per-file `stat()` scan and a cached
result keyed on how many FOV files currently exist (`NOTEBOOK_GUIDELINES.md`
#2-4) -- an actively-acquiring round gains files between notebook runs, so
the cache is invalidated once the file count grows rather than trusting a
stale one forever.

Two ways to estimate how long one frame actually takes: the THEORETICAL rate from
this round's HAL `<exposure_time>` (what `acquisition.dave.estimate_dave_experiment`
would use -- exposure time only, no stage-move/fluidics/readout overhead it doesn't
measure), and the EXPERIMENTAL rate measured directly from real file-write
timestamps -- the time between consecutive FOV files finishing (sorted by actual
write time via `common.io.path_mtime`, robust to on-disk listing order and
zarr-aware), divided by this round's frame count. The experimental rate captures
whatever real overhead the theoretical, exposure-time-only estimate can't see, so
section 9's trim-savings estimate below uses the **experimental** rate, not the
theoretical one.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached (NOTEBOOK_GUIDELINES.md #2/#3), keyed on how many FOV files currently
# exist -- an actively-acquiring round gains files between notebook runs, so
# the cache is invalidated once the file count grows rather than trusted forever.
timing_cache = cache_dir / f"timing_round{target_round_id}.json"

files_for_timing = [f for f in meta.files_for_round(target_round_id) if f.exists()]
if len(files_for_timing) < 2:
    raise ValueError(
        f"Need at least 2 written FOV files to measure real inter-FOV acquisition "
        f"timing -- only {len(files_for_timing)} found for round {target_round_id}."
    )

cached_timing = json.loads(timing_cache.read_text()) if timing_cache.exists() else None

if cached_timing is not None and cached_timing["n_files"] == len(files_for_timing):
    experimental_delta_s          = cached_timing["experimental_delta_s"]
    experimental_time_per_frame_s = cached_timing["experimental_time_per_frame_s"]
    theoretical_time_per_frame_s  = cached_timing["theoretical_time_per_frame_s"]
    delta_s = np.array(cached_timing["delta_s"])
    print(f"Loaded cached timing stats ({timing_cache.name}) -- "
          f"{len(files_for_timing)} FOV file(s), unchanged since it was written.")
else:
    reporter = ProgressReporter(total=len(files_for_timing), label="Reading FOV file mtimes")
    mtimes = [path_mtime(f) for f in reporter.wrap(files_for_timing)]
    mtimes = sorted(mtimes)
    delta_s = np.diff(mtimes)   # real wall-clock time between consecutive FOV-movies finishing

    # Median, not mean -- robust to the occasional outlier gap (a retry, a brief pause)
    # without needing to hand-filter anything.
    experimental_delta_s          = float(np.median(delta_s))
    experimental_time_per_frame_s = experimental_delta_s / len(frame_table)

    exposure_time_s = None
    for s in meta.series_for_round(target_round_id):
        if not s.hal_config:
            continue
        exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
        if exp is not None:
            exposure_time_s = exp
            break
    if exposure_time_s is None:
        exposure_time_s = 0.25
        print("WARNING: could not read <exposure_time> from this round's HAL config -- "
              "falling back to 0.25 s/frame (same fallback acquisition.dave.estimate_dave_experiment uses).")
    theoretical_time_per_frame_s = exposure_time_s

    timing_cache.write_text(json.dumps({
        "n_files":                       len(files_for_timing),
        "delta_s":                       delta_s.tolist(),
        "experimental_delta_s":          experimental_delta_s,
        "experimental_time_per_frame_s": experimental_time_per_frame_s,
        "theoretical_time_per_frame_s":  theoretical_time_per_frame_s,
    }))
    print(f"Measured timing from {len(files_for_timing)} FOV file(s); cached to {timing_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
print(f"Inter-FOV write-time delta (n={len(delta_s)} gaps, {len(files_for_timing)} FOV files): "
      f"median {experimental_delta_s:.2f}s, min {delta_s.min():.2f}s, max {delta_s.max():.2f}s, "
      f"std {delta_s.std():.2f}s")
print(f"Frames/FOV this round: {len(frame_table)}")
print(f"\nTheoretical  (HAL exposure_time only) : {theoretical_time_per_frame_s:.4f} s/frame")
print(f"Experimental (real file-write deltas)  : {experimental_time_per_frame_s:.4f} s/frame")
print(f"Experimental / theoretical ratio       : {experimental_time_per_frame_s / theoretical_time_per_frame_s:.2f}x "
      f"(> 1 means real per-frame time includes overhead the theoretical estimate misses)")
print("\n--> Using the EXPERIMENTAL rate for the time-savings estimate in section 9.")

## 9 — What-if: trim z-range acquisition

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1),
with `ProgressReporter` progress over the per-FOV trim computation and a
cached result keyed on `Z_MARGIN_UM`/`Z_MAX_TRIMMED_UM`/`THRESHOLD`/
`NTP_THRESHOLD`/FOV count (`NOTEBOOK_GUIDELINES.md` #2-4).

If a future acquisition only imaged, per FOV, up to `min(z_last_um + Z_MARGIN_UM,
Z_MAX_TRIMMED_UM)` instead of this round's full z-range, how much less disk space
and acquisition time would the round take? Uses this round's own `frame_table`
(section 3), `results_df` (section 6), and both `theoretical_time_per_frame_s`/
`experimental_time_per_frame_s` (section 8) directly -- no new image reads.

**`Z_MAX_TRIMMED_UM`** is an absolute cap on the trimmed depth, independent of
`Z_MARGIN_UM` -- `min(z_last_um + Z_MARGIN_UM, Z_MAX_TRIMMED_UM)` means the cap
wins even when the real measured signal plus margin would call for more depth.
A fixed default here is dangerous: if every FOV in a sample genuinely needs, say,
70 µm, a stale cap of 40 would make this cell report substantial "savings" that
are actually just deleted real tissue signal, with no indication anything was
truncated. So the default is derived from this round's own data instead of a
magic number -- this round's deepest measured `z_last_um` + `Z_MARGIN_UM` (or,
if no FOV had any detected signal at all, this round's own full imaged depth --
i.e. recommend no trimming when there's no basis to trim anything). That default
can never truncate below what was actually measured. Override it to a smaller,
deliberately-chosen value only if you have a real reason to (e.g. a known
physical/protocol limit) -- the calculation cell explicitly warns if the value
you set actually binds below any FOV's measured signal, so silent truncation
like the fixed-`40.0` case above can't happen unnoticed again.

Every color group in `frame_table` whose z varies (a real focus sweep -- including
any blank/return-to-bead-z frames, since their count can itself depend on total
sweep depth, e.g. `z_return_mode="progressive"`) is assumed to scale with the SAME
per-FOV z cutoff found for `CHANNEL_NM` -- i.e. every channel/color is assumed to
be imaging the same physical tissue volume, so a shallower `CHANNEL_NM` extent
implies a shallower everything-else extent too. Fixed (non-z-swept) frames -- e.g.
a single bead/reference shot -- are unaffected. An FOV with no detected signal at
all (`z_last_um` is `NaN`) is assumed to need 0 z-swept frames (i.e. could be
skipped entirely in the trimmed scheme).

**Both time-savings estimates are reported side by side** -- the THEORETICAL rate
(HAL exposure_time only) and the EXPERIMENTAL rate (measured directly from real
file-write timestamps, so it already includes whatever real per-frame overhead
exists: stage/z-move, fluidics, camera readout, ...) -- each for the whole round
AND for a single FOV file (one file's removed-frame count × that rate), so both
"what would the round save" and "what would one file save" are visible under
either assumption. Experimental is still the recommended one for planning, since
it reflects real overhead the theoretical estimate can't see. `N_ROUNDS_LIKE_THIS`
extrapolates this round's savings (both rates) to the whole experiment -- NOT
auto-assumed (different rounds can have different color/z-sweep configurations),
so it defaults to 1 (this round's own savings only); set it yourself if you know
how many rounds actually share this z-sweep depth.

In [ ]:
# Extra margin (um) kept beyond each FOV's measured z_last_um.
Z_MARGIN_UM = 3.0

# Absolute cap (um) on the trimmed depth, regardless of z_last_um. Defaults to
# this round's own deepest measured z_last_um + Z_MARGIN_UM -- i.e., by default
# the cap can NEVER truncate below what was actually measured for any FOV.
# Override to a smaller, deliberately-chosen value only if you have a real
# reason to cap depth below the measured maximum (e.g. a known physical/
# protocol limit) -- the calculation cell below warns explicitly if the value
# you set actually binds below any FOV's measured signal, so truncating real
# tissue silently (e.g. a stale fixed default like 40 when every FOV in the
# sample actually needs 70) can't happen unnoticed.
_z_last_um_max = results_df["z_last_um"].max()
Z_MAX_TRIMMED_UM = (
    float(_z_last_um_max + Z_MARGIN_UM) if pd.notna(_z_last_um_max)
    # No FOV had any detected signal at all -- fall back to this round's own
    # full imaged depth, i.e. recommend no trimming (the safe default when
    # there's no basis to trim anything).
    else float(frame_table["z"].max())
)

# How many rounds of the WHOLE experiment are assumed to share this round's
# z-sweep depth/configuration -- 1 = this round's own savings only. Set explicitly;
# not auto-derived from meta.n_rounds since different rounds can differ.
N_ROUNDS_LIKE_THIS = 1

print(f"Z_MARGIN_UM={Z_MARGIN_UM}, Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} "
      f"(auto-derived from this round's own data -- override above for a different cap), "
      f"N_ROUNDS_LIKE_THIS={N_ROUNDS_LIKE_THIS}")

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached (NOTEBOOK_GUIDELINES.md #2/#3), keyed on the parameters that actually
# affect the trim -- any of these changing invalidates the cache and triggers
# a recompute.
trim_cache      = cache_dir / f"zrange_trim_round{target_round_id}.csv"
trim_meta_cache = cache_dir / f"zrange_trim_round{target_round_id}.json"
trim_signature  = {
    "z_margin_um": Z_MARGIN_UM, "z_max_trimmed_um": Z_MAX_TRIMMED_UM,
    "threshold": float(THRESHOLD), "ntp_threshold": float(NTP_THRESHOLD),
    "n_fovs": len(results_df),
}

cached_trim_signature = json.loads(trim_meta_cache.read_text()) if trim_meta_cache.exists() else None

if cached_trim_signature == trim_signature and trim_cache.exists():
    trim_df = pd.read_csv(trim_cache)
    print(f"Loaded cached trim table ({trim_cache.name}) -- trim parameters/THRESHOLD/FOV count unchanged.")
else:
    # Every color group in frame_table, keyed by its (rounded) color -- NaN (blank/
    # no-laser) frames get a sentinel key so they form their own group rather than
    # being silently dropped by groupby. A group "is z-swept" if its frames actually
    # span more than one z value (a real focus sweep); everything else is a fixed,
    # unaffected frame (e.g. a single bead/reference shot).
    color_key = frame_table["color"].round(0)
    color_key = color_key.where(color_key.notna(), -1)

    zswept_groups = {}
    n_fixed_frames = 0
    for key, grp in frame_table.groupby(color_key):
        z_vals = grp["z"].to_numpy()
        if pd.Series(z_vals).nunique() > 1:
            zswept_groups[key] = z_vals
        else:
            n_fixed_frames += len(grp)

    n_zswept_frames = sum(len(v) for v in zswept_groups.values())
    print(f"Frame table: {len(frame_table)} frame(s)/FOV total -- {len(zswept_groups)} z-swept color "
          f"group(s) ({n_zswept_frames} frame(s)), {n_fixed_frames} fixed frame(s) unaffected by trimming.")

    def frames_kept_for_fov(z_needed):
        """Frame count kept under the trimmed scheme: every fixed frame, plus every
        z-swept-group frame at or below z_needed (0 z-swept frames if z_needed is None,
        i.e. no signal was detected in this FOV at all)."""
        if z_needed is None:
            return n_fixed_frames
        return n_fixed_frames + sum(int((z_vals <= z_needed).sum()) for z_vals in zswept_groups.values())

    trim_rows = []
    reporter = ProgressReporter(total=len(results_df), label="Computing per-FOV trim")
    for _, row in reporter.wrap(list(results_df.iterrows())):
        z_last   = row["z_last_um"]
        z_needed = min(z_last + Z_MARGIN_UM, Z_MAX_TRIMMED_UM) if pd.notna(z_last) else None
        n_trimmed = frames_kept_for_fov(z_needed)
        trim_rows.append({
            "fov_id":            row["fov_id"],
            "z_last_um":         z_last,
            "z_needed_um":       z_needed,
            "n_frames_current":  len(frame_table),
            "n_frames_trimmed":  n_trimmed,
            "n_frames_removed":  len(frame_table) - n_trimmed,
        })
    trim_df = pd.DataFrame(trim_rows)
    trim_df.to_csv(trim_cache, index=False)
    trim_meta_cache.write_text(json.dumps(trim_signature))
    print(f"Computed trim table for {len(trim_df)} FOV(s); cached to {trim_cache.name}.")

# Z_MAX_TRIMMED_UM is an absolute cap that wins over the real measurement --
# explicitly flag every time it actually binds below a FOV's own measured
# signal, so trimming that would delete real tissue can never happen
# unnoticed (checked on every run, cached or freshly computed).
_capped = (trim_df["z_last_um"] + Z_MARGIN_UM) > Z_MAX_TRIMMED_UM
n_capped = int(_capped.sum())
if n_capped > 0:
    max_truncated_um = float((trim_df["z_last_um"] + Z_MARGIN_UM - Z_MAX_TRIMMED_UM).clip(lower=0).max())
    print(f"\nWARNING: Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} um is BELOW z_last_um+Z_MARGIN_UM for "
          f"{n_capped}/{len(trim_df)} FOV(s) -- the trimmed scheme would cut off up to "
          f"{max_truncated_um:.1f} um of real measured tissue signal for those FOV(s). "
          f"Raise Z_MAX_TRIMMED_UM in the parameters cell above if this isn't intentional.")

In [ ]:
# ---- Display --------------------------------------------------------------
def format_bytes(n):
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024


# config.frame_width/frame_height are only needed to reshape raw .dax bytes
# (see common.io.iter_image_frames) -- .zarr/.tiff carry their own shape, so
# they're routinely left at their None default and can't be trusted here.
# Read one real frame directly instead, so this works regardless of format
# or whether config.frame_width/frame_height/image_size_px were ever set.
_sample_fpath = next(f for f in files if f.exists())
_sample_frame = next(frame for _, frame in iter_image_frames(
    _sample_fpath, [0], frame_width=config.frame_width, frame_height=config.frame_height,
))
frame_height_px, frame_width_px = _sample_frame.shape
frame_bytes = frame_width_px * frame_height_px * 2   # uint16 -- 2 bytes/pixel

n_fovs                          = len(trim_df)
bytes_saved_per_fov              = trim_df["n_frames_removed"] * frame_bytes
total_bytes_current_this_round   = n_fovs * len(frame_table) * frame_bytes
total_bytes_saved_this_round     = int(bytes_saved_per_fov.sum())

print(f"\n--- Round {target_round_id}, {n_fovs} FOV(s) ---")
print(f"Frames/FOV: {len(frame_table)} -> mean {trim_df['n_frames_trimmed'].mean():.1f} "
      f"({trim_df['n_frames_removed'].mean():.1f} removed/FOV on average)")
print(f"Space: {format_bytes(total_bytes_current_this_round)} -> "
      f"{format_bytes(total_bytes_current_this_round - total_bytes_saved_this_round)}  "
      f"(saved {format_bytes(total_bytes_saved_this_round)}, "
      f"{100 * total_bytes_saved_this_round / total_bytes_current_this_round:.1f}%)")

# Both time-savings estimates side by side (section 8): THEORETICAL (HAL
# exposure_time only) and EXPERIMENTAL (measured from real file-write deltas,
# so it already includes whatever real per-frame overhead exists). Each is
# reported for the whole round AND for a single FOV file (that file's own
# removed-frame count x the rate) -- "what would the round save" and "what
# would one file save" under either assumption.
time_rates = {
    "theoretical": theoretical_time_per_frame_s,
    "experimental": experimental_time_per_frame_s,
}
time_saved_per_fov_s = {}   # label -> pd.Series, one value per FOV
total_time_current_s = {}
total_time_saved_s   = {}

for label, rate in time_rates.items():
    time_saved_per_fov_s[label] = trim_df["n_frames_removed"] * rate
    total_time_current_s[label] = n_fovs * len(frame_table) * rate
    total_time_saved_s[label]   = float(time_saved_per_fov_s[label].sum())

    print(f"\nTime ({label}, {rate:.4f} s/frame):")
    print(f"  Round total:  {format_duration(total_time_current_s[label])} -> "
          f"{format_duration(total_time_current_s[label] - total_time_saved_s[label])}  "
          f"(saved {format_duration(total_time_saved_s[label])}, "
          f"{100 * total_time_saved_s[label] / total_time_current_s[label]:.1f}%)")
    print(f"  Single FOV file (average): {trim_df['n_frames_removed'].mean():.1f} frame(s) removed x "
          f"{rate:.4f} s/frame = {format_duration(time_saved_per_fov_s[label].mean())} saved")

print(f"\n--> Experimental is the recommended estimate for planning: it already reflects real "
      f"per-frame overhead (stage/z-move, fluidics, camera readout, ...) the theoretical "
      f"exposure-time-only rate misses.")

if N_ROUNDS_LIKE_THIS != 1:
    print(f"\n--- Extrapolated to {N_ROUNDS_LIKE_THIS} round(s) assumed to share this z-sweep "
          f"(N_ROUNDS_LIKE_THIS) ---")
    print(f"Space saved: {format_bytes(total_bytes_saved_this_round * N_ROUNDS_LIKE_THIS)}")
    for label in time_rates:
        print(f"Time saved ({label}): {format_duration(total_time_saved_s[label] * N_ROUNDS_LIKE_THIS)}")

trim_csv = config.analysis_dir / f"tissue_thickness_zrange_trim_round{target_round_id}.csv"
trim_df.to_csv(trim_csv, index=False)
print(f"\nSaved: {trim_csv}")

## 10 — Verify: per-FOV thumbnail mosaic at the last passing z

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

A visual sanity check on section 6's `z_last_um` calculation: for every FOV with
detected signal, render the ACTUAL frame at that FOV's own `z_last_um` -- not a
fixed z shared across every FOV, each tile comes from whatever frame/z that
specific FOV's calculated cutoff landed on -- then tile them into one mosaic
laid out by stage position (`analysis.round.create_mosaic`, the same tool the
online scheduler uses for real per-round mosaics).

Three things this section gets right that a naive per-tile rendering wouldn't:

- **Microscope camera orientation.** A raw camera frame is not necessarily
  already in the same orientation as the stage-position grid -- each
  microscope has its own `flip_horizontal`/`flip_vertical`/`transpose`
  camera->stage convention (read from `MERci/data/configs/merlin/microscope/`,
  the same files MERlin itself uses -- `MICROSCOPE` in section 2 selects
  which one). Every frame is re-oriented (`merlin_config.
  apply_microscope_orientation`, in MERlin's own order: transpose, then
  flip_horizontal, then flip_vertical) before being placed in the mosaic --
  skipping this makes the mosaic look rotated/transposed relative to the
  real tissue layout even though each individual tile is correct.
- **One shared intensity scale, not one per tile.** `create_thumbnail`'s usual
  per-frame percentile contrast-stretch (used elsewhere in this notebook)
  would auto-brighten/darken each tile independently -- exactly the opposite
  of what a cross-FOV comparison needs. This section caches raw (unstretched)
  downsampled pixel values per FOV and computes ONE percentile-based
  `vmin`/`vmax` pooled across every FOV, so brightness differences between
  tiles reflect real intensity differences, not independent auto-contrast.
- **A z label on every tile.** Each tile's top-left corner shows the
  `z_last_um` value it was rendered at, so the mosaic doubles as a legend for
  itself.

If `z_last_um` is being calculated correctly, every tile should look like a
real tissue edge (the deepest plane that still had signal) -- a tile that's
clearly blank/noise, or clearly mid-tissue rather than at an edge, would flag
a bug in the NTP-threshold logic worth investigating.

In [ ]:
# ---- Calculation --------------------------------------------------------
# This microscope's camera->stage orientation (MERlin's own convention --
# see MICROSCOPE_ORIENTATION_DIR/resolve_microscope_parameters_filename),
# applied to every raw frame below BEFORE downsampling/caching so the
# mosaic genuinely reflects the tissue's real layout instead of appearing
# rotated/transposed relative to the FOV grid.
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")

# Cached per-FOV RAW (not contrast-stretched) thumbnails -- a plain spatial
# downsample to config.thumbnail_size that keeps real relative pixel values,
# unlike analysis.fov.create_thumbnail's usual per-frame percentile stretch.
# Every FOV needs to share the SAME intensity scale in the display cell below
# for a meaningful cross-FOV comparison, which a per-tile auto-stretch would
# defeat. Cached as .npy (float32), one raw-pixel disk read per FOV
# (NOTEBOOK_GUIDELINES.md #2/#3), skipped once already rendered.
last_z_raw_thumbnails_dir = cache_dir / "last_z_raw_thumbnails" / f"round{target_round_id}"
last_z_raw_thumbnails_dir.mkdir(parents=True, exist_ok=True)


def last_z_raw_thumbnail_path(fov_id):
    return last_z_raw_thumbnails_dir / f"fov{fov_id:04d}.npy"


fov_ids_with_signal = results_df.loc[results_df["z_last_um"].notna(), "fov_id"].tolist()
to_render = [f for f in fov_ids_with_signal if not last_z_raw_thumbnail_path(f).exists()]
print(f"{len(fov_ids_with_signal)} FOV(s) with detected signal; "
      f"{len(fov_ids_with_signal) - len(to_render)} thumbnail(s) already cached, "
      f"{len(to_render)} to render.")

if to_render:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_render), label="Rendering last-z raw thumbnails")
    for fov_id in reporter.wrap(to_render):
        z_last    = float(results_df.loc[results_df["fov_id"] == fov_id, "z_last_um"].iloc[0])
        counters  = channel_counters[fov_id]
        pos       = int(np.argmin(np.abs(counters["z_um"] - z_last)))
        frame_idx = int(counters["frame_indices"][pos])

        fpath = next(f for f in files if meta.fov_id_of_file(f) == fov_id)
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        np.save(last_z_raw_thumbnail_path(fov_id), thumb.astype(np.float32))

In [ ]:
# ---- Display --------------------------------------------------------------
raw_thumbnails = {fov_id: np.load(last_z_raw_thumbnail_path(fov_id)) for fov_id in fov_ids_with_signal}

# One shared intensity scale across every FOV (not per-tile), so tiles are
# genuinely comparable -- same percentile-clip convention as elsewhere in
# this notebook (config.thumbnail_percentile_clip), computed over every
# pooled thumbnail pixel at once instead of per-frame.
pooled_pixels  = np.concatenate([t.ravel() for t in raw_thumbnails.values()])
lo_pct, hi_pct = config.thumbnail_percentile_clip
vmin, vmax     = np.percentile(pooled_pixels, [lo_pct, hi_pct])
print(f"Shared display scale (p{lo_pct:.0f}-p{hi_pct:.0f} over all {len(raw_thumbnails)} FOV thumbnails): "
      f"[{vmin:.0f}, {vmax:.0f}]")


def _to_uint8(thumb, vmin, vmax):
    scaled = (thumb.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)


last_z_thumbnails_uint8 = {fov_id: _to_uint8(t, vmin, vmax) for fov_id, t in raw_thumbnails.items()}
last_z_positions        = {fov_id: meta.fovs[fov_id].position for fov_id in fov_ids_with_signal}
last_z_labels           = {
    fov_id: f"{float(results_df.loc[results_df['fov_id'] == fov_id, 'z_last_um'].iloc[0]):.0f}"
    for fov_id in fov_ids_with_signal
}

last_z_mosaic_flip_y = resolve_round_flip_y(target_round_id, config, meta)
last_z_mosaic_path   = figures_dir / f"tissue_thickness_last_z_mosaic_round{target_round_id}.png"
create_mosaic(last_z_thumbnails_uint8, last_z_positions, last_z_mosaic_path,
              thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
              flip_y=last_z_mosaic_flip_y, labels=last_z_labels)
display_mosaic(last_z_mosaic_path, target_round_id)

n_missing_signal = len(results_df) - len(fov_ids_with_signal)
if n_missing_signal:
    print(f"({n_missing_signal} FOV(s) had no detected signal at all -- excluded from this "
          f"mosaic; see section 6's summary.)")